# ICS 235 Assignment 1
## Exploratory Data Analysis and the k-NN Classifier
# <span style="color:red">Due: 11:59 PM on Wednesday, September 23</span>

## Instructions

1. Name your notebook file using your last and first name.
    - For example, if your name is Harry Potter, rename the notebook file as `PotterHarry_1.ipynb` (the number at the end is the assignment number).
2. Only use the `.ipynb` file extension. Other formats (`.rtf`, `.zip`, `.docx`, `.pdf`) are not accepted.
3. The data file will be available to the instructor and the TA, so there is no need to upload it. Make sure you use the same filename given in the assignment.
4. Save the data file in your **working directory**.
5. **Do not modify or delete the provided code.**
6. Clean your code before submission.
    - If needed, provide clear documentation describing the purpose and use of every class or function you write.
    - Your submission should **show only the required outputs**.
7. **<font color='red'>IMPORTANT:</font>** Before submitting your homework:
   - Save the notebook.
   - Restart the kernel to clear its memory and **run the whole notebook top to bottom.** (`Kernel → Restart & Run All Cells`).
   - Confirm every cell completes without error and all outputs are visible.
8. Write your full name in the cell below.


### AI tools

You may use AI coding assistants as a learning aid, but **you must understand and be able to
explain every line you submit**. You may be asked to do so. Submitting code you cannot explain
is an academic integrity violation.
***

## Your Name: Denny Huang
***

# Getting Started

## About the data

This assignment uses `pa1_oahu_housing.csv`, a table of residential properties on Oʻahu.

> **This dataset is synthetic.** It was generated for instructional use in this course. The regions and the general shape of the relationships are realistic, but no row corresponds to a real property, and you should not draw conclusions about the actual Hawaiʻi housing market from it.

The table has **420 rows** and **9 columns**:

| Column | Description |
|---|---|
| `region` | District of Oʻahu (Honolulu, Kailua, Kaneohe, Kapolei, Mililani, North Shore) |
| `dist_downtown_km` | Straight-line distance from downtown Honolulu, in kilometers |
| `elevation_m` | Elevation of the property above sea level, in meters |
| `lot_sqft` | Total lot area, in square feet |
| `bedrooms` | Number of bedrooms |
| `bathrooms` | Number of bathrooms (half-baths appear as `.5`) |
| `age_years` | Age of the structure, in years |
| `living_sqft` | Interior living area, in square feet |
| `price_tier` | **The target.** One of `affordable`, `mid`, `luxury` |

## The task

Your job is to predict a property's `price_tier` from its other attributes, using the **k-Nearest Neighbors (k-NN)** classifier. Along the way you will do the work that often comes *before* a model: inspecting the data, finding its defects, and understanding why a distance-based algorithm cares so much about the units your features are measured in.

Download `pa1_oahu_housing.csv` from Lamaku and save the file in your **working directory**.

## Importing packages

Run the following cell. If any of these libraries are not installed on your machine, install
them with `pip install <name>` or `conda install <name>`.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Reproducibility: use this constant everywhere a random_state is required.
RANDOM_STATE = 42

## Loading the dataset

We load the dataset into a `pandas.DataFrame` using `pd.read_csv()`.

In [2]:
housing = pd.read_csv("pa1_oahu_housing.csv")

Let's confirm the file parsed correctly by looking at the first five rows.

In [3]:
housing.head()

,region,dist_downtown_km,elevation_m,lot_sqft,bedrooms,bathrooms,age_years,living_sqft,price_tier
0,Kaneohe,16.8,49,5608,3,3,29.0,1995.0,mid
1,Kaneohe,17.9,30,5466,3,3,35.0,NaN,mid
2,Honolulu,3.4,0,3979,3,2,33.0,2017.0,mid
3,Honolulu,6.2,25,4769,4,3,38.0,3115.0,luxury
4,Mililani,24.0,96,5322,2,2,41.0,1979.0,affordable


## A short refresher on pandas

You will need a few pandas operations throughout. Run the next three cells &mdash; they are here so you can copy the patterns later if needed.

**Selecting one column** gives you a `pandas.Series`:

In [4]:
living_area = housing["living_sqft"]
print(type(living_area))
living_area.head()

<class 'pandas.Series'>


0    1995.0
1       NaN
2    2017.0
3    3115.0
4    1979.0
Name: living_sqft, dtype: float64

**Selecting several columns** (note the list inside the brackets) gives you a `DataFrame`:

In [5]:
size_columns = housing[["bedrooms", "bathrooms", "living_sqft"]]
size_columns.head()

,bedrooms,bathrooms,living_sqft
0,3,3,1995.0
1,3,3,NaN
2,3,2,2017.0
3,4,3,3115.0
4,2,2,1979.0


**Boolean masks** let you filter rows. Combine conditions with `&` (and) and `|` (or) — *not*
the Python keywords `and`/`or` — and parenthesize each condition:

In [6]:
big_and_close = housing[(housing["living_sqft"] > 3000) & (housing["dist_downtown_km"] < 10)]
print(f"{len(big_and_close)} properties are larger than 3000 sqft and within 10 km of downtown")

8 properties are larger than 3000 sqft and within 10 km of downtown


***
# Exercises

This assignment has **three parts**, worth 100 points total.

Write your answer wherever you see a *`# YOUR CODE`* comment in a code cell or a **`YOUR ANSWER:`** prompt in a markdown cell. Some cells already contain code to get you started — **do not modify the provided code**. The point value of each question is given in parentheses after the question text.

| Part | Topic | Points |
|---|---|---|
| 1 | Preliminary analysis and data quality | 35 |
| 2 | k-NN with scikit-learn: scaling and choosing *k* | 40 |
| 3 | Final evaluation and data story | 25 |

***
## Part 1: Preliminary Analysis and Data Quality (35 points)

Before fitting any model, find out what you actually have. The questions in this part cover some checks you should routinely perform when working with any new dataset.

### Question 1. Dataset structure (1 $\times$ 4 = 4 points)

1. Print the number of rows and the number of columns, as a readable sentence
   (e.g. `"The dataset has ___ rows and ___ columns."`).
2. Display the column names together with their data types and non-null counts.
   - Hint: `DataFrame.info()` does all of this in one call.
3. Display the **first 10 rows**.
4. In the answer cell below, state how many columns are **numerical** and how many are
   **categorical**, and identify which column is the **target**.

In [22]:
# YOUR CODE
rows, cols = housing.shape
print(f"This dataset has {rows} rows and {cols} columns")
print(housing.info())
housing.head(10)


This dataset has 420 rows and 9 columns
<class 'pandas.DataFrame'>
RangeIndex: 420 entries, 0 to 419
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   region            420 non-null    str    
 1   dist_downtown_km  420 non-null    float64
 2   elevation_m       420 non-null    int64  
 3   lot_sqft          420 non-null    int64  
 4   bedrooms          420 non-null    int64  
 5   bathrooms         420 non-null    int64  
 6   age_years         410 non-null    float64
 7   living_sqft       408 non-null    float64
 8   price_tier        420 non-null    str    
dtypes: float64(3), int64(4), str(2)
memory usage: 29.7 KB
None


,region,dist_downtown_km,elevation_m,lot_sqft,bedrooms,bathrooms,age_years,living_sqft,price_tier
0,Kaneohe,16.8,49,5608,3,3,29.0,1995.0,mid
1,Kaneohe,17.9,30,5466,3,3,35.0,NaN,mid
2,Honolulu,3.4,0,3979,3,2,33.0,2017.0,mid
3,Honolulu,6.2,25,4769,4,3,38.0,3115.0,luxury
4,Mililani,24.0,96,5322,2,2,41.0,1979.0,affordable
5,Kapolei,35.7,29,3702,4,4,12.0,3358.0,luxury
6,Honolulu,1.9,21,4823,3,2,51.0,2255.0,mid
7,Honolulu,0.5,43,3472,3,2,NaN,1898.0,mid
8,Mililani,20.4,103,8290,3,3,35.0,2524.0,luxury
9,Honolulu,5.9,13,5624,3,3,43.0,2412.0,mid


**YOUR ANSWER (Q1.4):**

Categorical: 2 categorical columns: region, price_tier <br>
Numerical: 7 numerical columns: dist_downtown_km, elevation_m, lot_sqft, bedrooms, bathrooms, age_years, living_sqft <br>
Target: price_tier

### Question 2. The target distribution and the baseline you must beat (1 + 1 + 3 + 3 = 8 points)

1. Print how many properties fall into each `price_tier`.
2. Print the same information as **proportions** rather than counts. Do not hardcode the numbers from Problem 1 to calculate the proportions. Display the proportions to three decimal places (e.g., 0.234).<br>
   - Hint: Check whether the parameters of the function that you used in Problem 1 can help.
3. A trivial "classifier" could ignore every feature and always predict the **most frequent class**. Compute that classifier's *accuracy* on this dataset **in code** (do not just state it) and print it.
4. In the answer cell, explain what this number means and why every model you build later has to be compared against it.

In [42]:
# YOUR CODE
properties_by_tier = housing['price_tier'].value_counts()
print(properties_by_tier)

properties_by_tier = housing['price_tier'].value_counts(normalize=True)
print(properties_by_tier)

properties_by_tier = housing['price_tier'].value_counts(normalize=True).max()
print(f"Accuracy: {properties_by_tier}")



price_tier
mid           140
luxury        140
affordable    140
Name: count, dtype: int64
price_tier
mid           0.333333
luxury        0.333333
affordable    0.333333
Name: proportion, dtype: float64
Accuracy: 0.3333333333333333


>**YOUR ANSWER (Q2.4):** This number means that if we ignored every featurea nd always predicted the most frequent class, we would still be right 33.33% of the time. Every model that is built later has to be compared against it because this the minimum accuracy any model that is built should beat.
>
>

### Question 3. Missing values (1 + 2 + 2 + 3 = 8 points)

1. Print the number of missing values in each column.
2. Print the **percentage** of missing values, showing *only the columns that actually have missing data*.
   - Hint: `DataFrame.isnull().sum()` counts them; dividing by `len(df)` converts to a fraction.
3. Compute and print how many **rows** contain at least one missing value, and what percentage of the dataset that is.
   - Hint: `DataFrame.isnull().any(axis=1)` gives one boolean per row.
4. In the answer cell: if you dropped every row containing a missing value, how many rows would you be left with? When would this not be an acceptable trade-off, and what is the alternative? Justify your answer.

In [ ]:
# YOUR CODE



>**YOUR ANSWER (Q3.4):**
>
>

### Question 4. Inconsistent categories (2 + 1 + 2 + 3 = 8 points)

Categorical columns typed by hand are almost never clean.

1. Print the value counts of the `region` column. How many **distinct labels** does pandas report?
2. Look carefully at the labels. Identify the ones that refer to the **same real region** but were recorded differently, and state which is which in the answer cell.
3. Create a **cleaned copy** of the DataFrame (call it `housing_clean`) in which the `region` values are standardized. Do **not** overwrite `housing`.
   - Hint: `.str.strip()` removes stray whitespace and `.str.title()` normalizes capitalization.
   - Re-print the value counts afterwards and report the new number of distinct labels.
4. In the answer cell: what could have gone wrong if you had skipped this step and applied one-hot encoding directly? In particular, what impact would this have on a distance-based model such as a k-NN classifier?

In [ ]:
# YOUR CODE



>**YOUR ANSWER:**
>
>**Q4.2:**
>
>
>**Q4.4:**
>
>

### Question 5. Numeric ranges &mdash; a warning sign for k-NN (1 + 3 + 3 = 7 points)

1. Create a list named `numeric_features` containing the names of all numerical columns. Then, use this list to display the summary statistics for those columns in `housing_clean` using `describe()`.
2. Create and display a small DataFrame named `ranges` with one row for each numerical feature. It should contain the feature's **minimum**, **maximum**, and **range** (maximum − minimum), sorted from largest range to smallest.
   - Hint: Use `DataFrame.sort_values()` to sort by a specific column.
3. In the answer cell, identify the feature with the largest range and estimate how many times larger its range is than the smallest. Why should this concern you before running k-NN?

In [ ]:
# YOUR CODE



>**YOUR ANSWER (Q5.3):**
>
>

***
## Part 2: k-NN with scikit-learn &mdash; Scaling and Choosing *k* (40 points)

Now we build the model. You will use scikit-learn's `KNeighborsClassifier` throughout &mdash; you are not asked to implement the algorithm yourself.

Everything goes inside a `Pipeline`, so that every transformation is refit from scratch on each cross-validation fold. Run the provided setup cell first.

In [ ]:
# ============================ PROVIDED SETUP — do not modify ============================
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier

# Features = everything except the target. Target = price_tier.
X = housing_clean.drop(columns="price_tier")
y = housing_clean["price_tier"]

# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y)

print(f"Training set: {X_train.shape[0]} rows    Test set: {X_test.shape[0]} rows")
print(f"Training class balance:\n{y_train.value_counts()}")
# =======================================================================================

### Question 1. Build a leak-free preprocessing pipeline (4 + 2 + 2 + 3 = 11 points)

1. Build a `ColumnTransformer` named `preprocessor` that applies:
   - to the **numerical columns**: a `SimpleImputer(strategy="median")` followed by a `StandardScaler()` &mdash; chain them with a small inner `Pipeline`;
   - to the **`region` column**: a `OneHotEncoder(handle_unknown="ignore")`.
2. Wrap `preprocessor` and a `KNeighborsClassifier(n_neighbors=11)` in a single `Pipeline` named `knn_pipeline`.
3. Fit it on the training data and print its accuracy on **both** the training and the test set.
4. In the answer cell:
   - Explain why the imputer and the scaler must live **inside** the pipeline, rather than being applied to the whole DataFrame before splitting. What specifically would go wrong?
   - Explain what `handle_unknown="ignore"` protects you against.

In [ ]:
# YOUR CODE



>**YOUR ANSWER (Q1.4):**
>
>

### Question 2. Does scaling actually matter? (3 + 4 + 4 = 11 points)

In Part 1 you noticed that `lot_sqft` has a range roughly 2,650× larger than `bathrooms`. Now
measure what that does to the model.

1. Build a second pipeline, `knn_pipeline_unscaled`, identical to the first **except that the `StandardScaler` step is removed** (keep the imputer and the one-hot encoder).
2. For **both** pipelines, report:
   - the mean **5-fold cross-validation accuracy** on the training data, and
   - the accuracy on the test set.
   - Hint: `cross_validate(pipeline, X_train, y_train, cv=5)` returns a dict; the validation scores are under the key `"test_score"`.
3. In the answer cell:
   - Describe how large the effect of removing the scaler is.
   - Using the feature ranges from Part 1, Question 5, explain *why* removing the scaler hurts k-NN performance.
   - Which features does the unscaled model give the most influence to, and which features does it effectively downweight?

In [ ]:
# YOUR CODE



>**YOUR ANSWER (Q2.3):**
>
>

### Question 3. Choosing *k* by cross-validation (4 + 2 + 3 + 3 = 12 points)

`k` is a **hyperparameter**: it is not learned by fitting, it is chosen by us. We choose it using **validation** performance, never the test set.

1. For every `k` in the provided list, run **10-fold cross-validation** on the **training data** using the **scaled pipeline**, and record both the mean **training** score and the mean **validation** score.
   - Hint: pass `return_train_score=True` to `cross_validate`; the two keys you need are `"train_score"` and `"test_score"`. Note that scikit-learn calls the held-out fold the "test" score even though it is functioning as a validation set here.
2. Report the value of `k` with the highest mean validation accuracy, and that accuracy. Store it in a variable named `best_k` &mdash;  you will need it in Part 3.
3. **Plot both curves on a single figure**: `k` on the x-axis, accuracy on the y-axis, one line for mean training accuracy and one for mean validation accuracy, in different colors, with a legend, axis labels and an appropriate title. Mark the best `k` (e.g., with a vertical dashed line).
4. In the answer cell, describe the **shape** of the two curves: does the model overfit or underfit? If so, where and how can you tell from the *gap* between the two lines?

In [ ]:
# ======= PROVIDED SETUP — do not modify =======
k_values = list(range(1, 60, 2))
# ==============================================

In [ ]:
# YOUR CODE



>**YOUR ANSWER (Q3.4):**
>
>

### Question 4. Conceptual questions about *k* (2 $\times$ 3 = 6 points)

Answer each in the markdown cell below. You may verify your answers in code first, but the answer cell must explain the *reason*, not just report the value.

1. When `k = 1`, the **training** accuracy is exactly 1.0. Why must this be true &mdash; and why is it not evidence that the model is good? 
2. Suppose you set `k` equal to the number of training samples. What will the model predict for *any* input, and what accuracy would that give on our test set? (Look carefully at the class balance of the training set before answering.) 
3. In terms of **bias** and **variance**, which end of the `k` range is which? 

In [ ]:
# Optional: verify your answers to (1) and (2) in code before writing them up.
# YOUR CODE



<!--SOLUTION-->
>**YOUR ANSWER:**
>
>**Q4.1:**
>
>
>**Q4.2:**
>
>
>**Q4.3:**
>
>

***
## Part 3: Final Evaluation and Data Story (25 points)

You have cleaned the data and selected `k` using *validation* data. The test set has not been touched. Now &mdash; and **only** now &mdash; you may use it, **once**.

### Question 1. Final evaluation on the test set (3 + 2 + 2 + 3 = 10 points)

1. Build a final pipeline using the **`best_k`** you found in Part 2, Question 3, and fit it on the **full training set**. 
2. Predict on the test set and report the **test accuracy**, alongside the baseline from Part 1. 
3. Display the **confusion matrix** for the test set predictions.
   - Label both the rows and columns with the tier names in the following order:
      `affordable`, `mid`, `luxury`<br>
     **The order matters** because it determines how the rows and columns of the confusion matrix are arranged. Do not use alphabetical ordering. 
   - Hint: Use `confusion_matrix(y_test, y_pred, labels=tiers)`, where `tiers` contains the class names in the required order. Then pass `tiers` as `display_labels` when creating the `ConfusionMatrixDisplay`.
4. In the answer cell: 
   - Is there any tier the model handles noticeably worse than the others?
   - Which pair(s) of classes are confused most often?
   - Is accuracy a sufficient summary of this model's performance? Why or why not?

In [ ]:
# YOUR CODE



>**YOUR ANSWER (Q1.4):**
>
>

### Question 2. Your data story (15 points)

Write **200–300 words**, in plain prose, that an interested non-expert could follow. Do not include code. Your response should tell the story of your analysis and address **all four** of the following:

1. **The data and the question:** What is in this dataset, and what question were you trying to answer with it?
2. **Cleaning and preparation:** What did you have to clean, fix, or prepare before modeling? Explain **why each step mattered** for the analysis.
3. **Choosing `k`:** How did you choose the value of `k` for your final model? Explain why you could **not simply choose the value of `k` that gave the highest test accuracy**.
4. **Model performance:** How well does the final model perform compared with a sensible baseline? Describe **where the model performs well and where it fails**, using evidence from your results.

>**YOUR ANSWER:**
>
>